## Instructions

1. Do not write your name on the assignment. Be careful about any warnings that might display some file path with your name included.

2. You may talk to a friend, discuss the questions and potential directions for solving them. However, you need to write your own solutions and code separately, and not as a group activity.

3. You are expected to carefully read and comply with the [generative AI policy](https://canvas.northwestern.edu/courses/233157/pages/generative-ai-policy?module_item_id=3421526) of this course.

4. Write your code in the *Code* cells and print the instructed output. If you are instructed to explain something in words, write your answer in the *Markdown* cells of the Jupyter notebook. Ensure that the solution is coded and/or written neatly enough to understand and grade.

5. Use [Quarto](https://quarto.org/docs/output-formats/html-basics.html) to render the *.ipynb* file as HTML. You will need to open the command prompt, navigate to the directory containing the file, and use the command: `quarto render filename.ipynb --to html`. Submit the HTML file.

6. This assignment is worth 100 points and is due on **June 9, 2025 at 11:59 pm**. 

7. **Five points given for properly formatting the assignment**. The breakdown is as follows:
- The submission must be an HTML file rendered using Quarto. (1 point).
- Your name should not be visible in the HTML file (including the file path in any warning that your code returns). (1 point)
- There are not excessively long outputs of extraneous information. (e.g. no printouts of entire data frames without good reason; there are not long printouts of which iteration a loop is on; there are not long sections of commented-out code, etc.) (1 point)
- Final answers for each question are written in Markdown cells. (1 point).
- There is no piece of unnecessary / redundant code, and no unnecessary / redundant text. (1 point)

## 1) Conceptual Questions (9 points)

### a)

Is it possible for an ensemble model (Voting or Stacking) to perform worse than one or more of its base models? **(1 point)** Why or why not? **(4 points)**

A voting ensemble can definitely underperform one of its base models since it either averages the predictions/probabilities of the base models or takes the modal prediction, meaning a voting ensemble's performance could be worsened by less accurate base models despite there being a very good base model. A stacking ensemble may underperform one of its base models when its meta model either overfits the predictions or assigns too much weight to bad base models and not enough weight to good base models.

### b)

If the ensemble model (Voting or Stacking) does perform worse than one or more of its base models, then what should be the course of action? **(4 points)**

You should either remove or reduce the weight ofthe worse performing base models from the estimators input set for the ensemble model, to more heavily weight the better-performing base models.

## 2) Regression with Ensembles (45 points)

In this question, you will use the **miami_housing.csv** file. You can find the description for the variables [here](https://www.kaggle.com/datasets/deepcontractor/miami-housing-dataset).

The `SALE_PRC` variable is the regression response and the rest of the variables, except `PARCELNO`, are the predictors.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict, cross_validate
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, StratifiedKFold, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import VotingClassifier, VotingRegressor, StackingRegressor, StackingClassifier
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.ensemble import BaggingRegressor, BaggingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, ElasticNet

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [3]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


### a)

Read the dataset. Create the training and test sets with a 60%-40% split and `random_state = 1`. **(1 point)**

In [4]:
miami_data = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/miami-housing.csv")
predictors = miami_data.drop(columns = ["SALE_PRC", "PARCELNO"], axis=1)
target = miami_data["SALE_PRC"]
X_train, X_test, y_train, y_test = train_test_split(predictors, target, test_size=0.4, random_state=1)

### b)

Recreate and train all the following **tuned** regression models: **(5x0.5 = 2.5 points)** **Set all `random_state`s to 1.**

- Bagged Trees from Assignment 3
- Random Forest from Assignment 3
- AdaBoost from Assignment 4
- Gradient Boosting from Assignment 4
- XGBoost from Assignment 4

Note that there will not be any cross-validation, since the models are already tuned; i.e. created with the best hyperparameters found in the previous assignments.

Print the test MAE of all five tuned models. **(5x0.5 = 2.5 points)**

In [5]:
xgb_4 = XGBRegressor(
    random_state=1,
    objective = 'reg:squarederror',
    colsample_bytree=0.5, 
    gamma=0,
    learning_rate=0.01,
    max_depth=6,
    n_estimators=3000,
    reg_lambda=0.01,
    subsample=1
).fit(X_train, y_train)
y_pred_xgb = xgb_4.predict(X_test)
print("XGB MAE:", mean_absolute_error(y_test, y_pred_xgb))

gbm_4 = GradientBoostingRegressor(
    random_state=1,
    loss='huber',
    learning_rate=0.1,
    max_depth=4,
    n_estimators=1750,
    subsample=0.75
).fit(X_train, y_train)
y_pred_gbm = gbm_4.predict(X_test)
print("GBM MAE:", mean_absolute_error(y_test, y_pred_gbm))

adaboost_4 = AdaBoostRegressor(
    random_state=1,
    estimator = DecisionTreeRegressor(random_state = 1, max_depth = 20),
    n_estimators = 127,
    learning_rate = 1).fit(X_train, y_train)
y_pred_adaboost = adaboost_4.predict(X_test)
print("AdaBoost MAE:", mean_absolute_error(y_test, y_pred_adaboost))

rf_3 = RandomForestRegressor(
    random_state=1,
    n_estimators = 200,
    bootstrap = True,
    max_features = 0.3,
    max_samples = 0.9).fit(X_train, y_train)
y_pred_rf = rf_3.predict(X_test)
print("RF MAE:", mean_absolute_error(y_test, y_pred_rf))

bagged_trees3 = BaggingRegressor(
    random_state=1,
    estimator = DecisionTreeRegressor(random_state =1),
    n_estimators = 250, 
    oob_score= True,
    bootstrap = True,
    bootstrap_features = False,
    max_features = 0.75,
    max_samples = 0.1
).fit(X_train, y_train)
y_pred_bagged_trees = bagged_trees3.predict(X_test)
print("Bagged Trees MAE:", mean_absolute_error(y_test, y_pred_bagged_trees))

XGB MAE: 41405.96724862058
GBM MAE: 43935.72032369877
AdaBoost MAE: 44708.14507607808
RF MAE: 45474.64608315603
Bagged Trees MAE: 55583.50580776362


### c)

Train a voting ensemble regressor with all of the five models from Part b. Note that all of the models are already tuned separately, which means the voting ensemble is already tuned (in a greedy way). Print the test MAE of the Voting Ensemble. **(5 points)** Is it better than all the base models? **(1 point)**

In [6]:
voting_reg = VotingRegressor(
    estimators=[
        ('xgb', xgb_4),
        ('gbm', gbm_4),
        ('adaboost', adaboost_4),
        ('rf', rf_3),
        ('bagged_trees', bagged_trees3)
    ],
    n_jobs=-1
).fit(X_train, y_train)
y_pred_voting = voting_reg.predict(X_test)
print("Voting Regressor MAE:", mean_absolute_error(y_test, y_pred_voting))

Voting Regressor MAE: 42743.24774578754


The voting ensemble is better than all but one of the base models (XGBoost).

### d) 

Retrain the voting ensemble regressor with the two models that return the lowest MAE in Part b. Print the test MAE and compare it with the MAEs in Parts b and c. **(3 points)**

In [7]:
voting_reg_d = VotingRegressor(
    estimators = [
        ('xgb', xgb_4),
        ('gbm', gbm_4)
    ],
    n_jobs=-1
).fit(X_train, y_train)
y_pred_voting_d = voting_reg_d.predict(X_test)
print("Voting Regressor MAE:", mean_absolute_error(y_test, y_pred_voting_d))

Voting Regressor MAE: 41291.410989223994


This voting regressor achieved a lower MAE than all of the base models and the voting ensemble from Part c.

### e)

Train a stacking ensemble regressor with all of the five models from Part b and Linear Regression as the meta model. Use 5-fold cross-validation with a shuffle (`random_state=1`) for the base model predictions. Print the test MAE. **(3 points)** Explain why the stacking ensemble in this question is not required to be tuned. **(2 points)**

**Hint:** Using the efficient implementation covered in lecture can save **a lot** of time.

In [8]:
cv_settings_meta = KFold(n_splits=5, shuffle=True, random_state=1)

pred_cv1 = cross_val_predict(xgb_4, X_train, y_train, cv=cv_settings_meta)
pred_cv2 = cross_val_predict(gbm_4, X_train, y_train, cv=cv_settings_meta)
pred_cv3 = cross_val_predict(adaboost_4, X_train, y_train, cv=cv_settings_meta)
pred_cv4 = cross_val_predict(rf_3, X_train, y_train, cv=cv_settings_meta)
pred_cv5 = cross_val_predict(bagged_trees3, X_train, y_train, cv=cv_settings_meta)

stacking_reg_df = pd.DataFrame({
    'xgb': pred_cv1,
    'gbm': pred_cv2,
    'adaboost': pred_cv3,
    'rf': pred_cv4,
    'bagged_trees': pred_cv5,
})

meta_model = LinearRegression().fit(stacking_reg_df, y_train)
stacking_test_df = pd.DataFrame({
    "xgb": y_pred_xgb,
    "gbm": y_pred_gbm,
    "adaboost": y_pred_adaboost,
    "rf": y_pred_rf,
    "bagged_trees": y_pred_bagged_trees,
})
y_pred_stacking = meta_model.predict(stacking_test_df)
print("Stacking Regressor MAE:", mean_absolute_error(y_test, y_pred_stacking))

Stacking Regressor MAE: 41236.696182966596


### f)

Print the weights of the base models in the stacking ensemble from Part e. **(2 points)**

In [9]:
print(pd.DataFrame({
    'model': ['xgb', 'gbm', 'adaboost', 'rf', 'bagged_trees'],
    'weight': meta_model.coef_
}))

          model    weight
0           xgb  0.709915
1           gbm  0.467015
2      adaboost  0.012853
3            rf  0.014479
4  bagged_trees -0.206041


### g)

Train **and tune** a stacking ensemble regressor with all of the five models from Part b and Lasso as the meta model. Use 5-fold cross-validation (CV) with a shuffle (`random_state=1`) both for the base model predictions and while tuning the regressor (or its parts that need to be tuned). Use `[0.001, 0.01, 0.1, 1, 10, 100]` as the array of hyperparameter values for cross-validation. 

Print the best average CV MAE and the test MAE.

**Hint:** Using the efficient implementation covered in lecture can save **a lot** of time.

 **(6 points)**

In [10]:
# use cv_settings_meta, stacking_reg_df, and stacking_test_df created earlier
meta_model_g = Lasso()
grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
gscv_g = GridSearchCV(
    meta_model_g,
    grid,
    cv = cv_settings_meta,
    scoring = 'neg_mean_absolute_error',
    n_jobs = -1
)

gscv_g.fit(stacking_reg_df, y_train)
print("best alpha:", gscv_g.best_params_['alpha'])
print("best CV MAE:", -gscv_g.best_score_)

final_meta = gscv_g.best_estimator_
y_pred_g = final_meta.predict(stacking_test_df)
print("Final Stacking Regressor MAE:", mean_absolute_error(y_test, y_pred_g))

/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.367e+13, tolerance: 6.784e+10
  model = cd_fast.enet_coordinate_descent(
/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.431e+13, tolerance: 7.027e+10
  model = cd_fast.enet_coordinate_descent(
/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, c

best alpha: 100
best CV MAE: 43006.97359372456
Final Stacking Regressor MAE: 41236.74228403558


/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.436e+13, tolerance: 6.933e+10
  model = cd_fast.enet_coordinate_descent(
/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.359e+13, tolerance: 6.784e+10
  model = cd_fast.enet_coordinate_descent(
/Users/vaibhavrangan/Downloads/Stat_303-3/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, c

### h)

Print the weights of the base models in the stacking ensemble from Part g. **(1 point)**

In [11]:
model_weights = pd.DataFrame({
    'model': ['xgb', 'gbm', 'adaboost', 'rf', 'bagged_trees'],
    'weight': final_meta.coef_
})
print(model_weights)

          model    weight
0           xgb  0.709943
1           gbm  0.467011
2      adaboost  0.012874
3            rf  0.014418
4  bagged_trees -0.206023


### i)

Train **and tune** a stacking ensemble regressor with all of the five models from Part b and a decision tree as the meta model. Use 5-fold cross-validation (CV) with a shuffle (`random_state=1`) both for the base model predictions and while tuning the regressor (or its parts that need to be tuned). Use `max_depth` values from 2 to 10 (inclusive) as the array of hyperparameter values for cross-validation. 

Print the best average CV MAE and the test MAE.

**Hint:** Using the efficient implementation covered in lecture can save **a lot** of time.

 **(6 points)**

In [12]:
# use cv_settings_meta, stacking_reg_df, and stacking_test_df created earlier
meta_model_i = DecisionTreeRegressor(random_state =1)
grid_i = {"max_depth": range(2,11)}
gscv_i = GridSearchCV(
    meta_model_i,
    grid_i,
    cv = cv_settings_meta,
    scoring = 'neg_mean_absolute_error',
    n_jobs = -1
)
gscv_i.fit(stacking_reg_df, y_train)
print("best max_depth:", gscv_i.best_params_['max_depth'])
print("best CV MAE:", -gscv_i.best_score_)

final_meta_i = gscv_i.best_estimator_
y_pred_i = final_meta_i.predict(stacking_test_df)
print("Final MAE:", mean_absolute_error(y_test, y_pred_i))

best max_depth: 6
best CV MAE: 47269.07138118447
Final MAE: 44241.35543996979


### j)

Print the importances of the base models in the stacking ensemble from Part i. **(2 points)**

In [13]:
print(pd.DataFrame({
    'model': ['xgb', 'gbm', 'adaboost', 'rf', 'bagged_trees'],
    'weight': final_meta_i.feature_importances_
}))

          model    weight
0           xgb  0.871866
1           gbm  0.119684
2      adaboost  0.001459
3            rf  0.004843
4  bagged_trees  0.002148


### k)

Compare the weights and importances found in Parts f, h, and j. Considering the base model performances in Part b, do they make sense? **(2 points)**

The weights of the Linear Regression and Lasso models are the same, which makes sense because their CV and test MAEs are nearly identical. The model from part i, with the DecisionTreeRegressor as the meta model, performs the worst but perplexingly, places the highest weight on the base model with the best performance, the XGBRegressor. I think the performance gap can be explained by the fact that the DecisionTreeRegressor functionally ignores the adaboost, random forest, and bagged trees models and greatly reduces the importance of the Gradient Boosting Regressor. The DecisionTreeRegressor thus discards the predictive power of all the models besides XGBoost, reducing its accuracy.

### l)

Use the three stacking ensemble models created in Parts e, g, and i to train an "ensemble of ensembles". It will use the three stacking ensemble models as base models for a voting ensemble regressor. **(5 points)** Print the test MAE. **(1 point)**

In [14]:
# we already have the test predictions from all 3 stacking models, so just average them like a Voting Regressor would
ens_pred_test = (y_pred_stacking + y_pred_g + y_pred_i) / 3
print("Ensemble of Ensembles MAE:", mean_absolute_error(y_test, ens_pred_test))

Ensemble of Ensembles MAE: 41283.41824714208


## 3) Classification with Ensembles (41 points)

In this question, you will use the **train.csv** and **test.csv** files. Each observation is a marketing call from a banking institution. The `y` variable is the classification response and represents whether the client subscribed for a term deposit (1) or not (0).

The predictors are `age`, `day`, `month`, and `education`.

### a)

Preprocess the data:

- Read the files. Create the training and the test datasets.
- Convert the response to 1s and 0s.
- One-hot-encode the categorical predictors (**Do not use `drop_first`**).

**(1 point)**

In [15]:
from sklearn.preprocessing import OneHotEncoder


train = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/train.csv")
test = pd.read_csv("/Users/vaibhavrangan/Downloads/Stat_303-3/Datasets/test.csv")

train["y"] = train["y"].apply(lambda x: 1 if x == "yes" else 0)
test["y"] = test["y"].apply(lambda x: 1 if x == "yes" else 0)

X_train = train.drop(columns=["y"], axis=1)
y_train = train["y"]
X_test = test.drop(columns=["y"], axis=1)
y_test = test["y"]


categorical_vars = ["education", "month"]

X_train_numerical = X_train.drop(columns=categorical_vars, axis=1)
X_test_numerical = X_test.drop(columns=categorical_vars, axis=1)

encoder = OneHotEncoder(sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[categorical_vars])
X_test_encoded = encoder.transform(X_test[categorical_vars])

X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_vars))
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(categorical_vars))
X_train_cleaned = pd.concat([X_train_numerical, X_train_encoded], axis=1)
X_test_cleaned = pd.concat([X_test_numerical, X_test_encoded], axis=1)


### b)

Train a hard voting ensemble classifier with all the following **tuned** models and their **tuned** thresholds:

- Random Forest from Assignment 3
- LightGBM from Assignment 4
- CatBoost from Assignment 4

Print the test accuracy and test recall of the voting ensemble. **(8 points)**

In [23]:
from sklearn.metrics import recall_score, accuracy_score

rf_class = RandomForestClassifier(
    random_state=1,
    n_estimators = 200,
    bootstrap = True,
    max_features = 0.05,
    max_samples = 0.3
).fit(X_train_cleaned, y_train)
thr1 = 0.13
y_pred_proba_rf = rf_class.predict_proba(X_test_cleaned)[:, 1]
y_pred_rf = (y_pred_proba_rf > thr1).astype(int)

lgbm_class = LGBMClassifier(
    random_state=1,
    verbosity =-1,
    colsample_bytree=0.5,
    learning_rate=0.01,
    max_depth = 8,
    n_estimators = 2500,
    reg_lambda = 1,
    subsample = 0.6
).fit(X_train_cleaned, y_train)
thr2 = 0.43
y_pred_proba_lgbm = lgbm_class.predict_proba(X_test_cleaned)[:, 1]
y_pred_lgbm = (y_pred_proba_lgbm > thr2).astype(int)

cat_class = CatBoostClassifier(
    random_state=1,
    verbose = False,
    scale_pos_weight = 7.53,
    learning_rate = 0.1,
    max_depth = 6,
    n_estimators = 300,
    reg_lambda = 1,
    subsample = 1
).fit(X_train_cleaned, y_train, verbose=0)
thr3 = 0.43
y_pred_proba_cat = cat_class.predict_proba(X_test_cleaned)[:, 1]
y_pred_cat = (y_pred_proba_cat > thr3).astype(int)

sum_preds = y_pred_rf + y_pred_lgbm + y_pred_cat
hard_vote = (sum_preds >= 2).astype(int)
print(accuracy_score(y_test, hard_vote))
print(recall_score(y_test, hard_vote))


0.7914302484836627
0.5497470489038786


### c)

Using the same base models as in Part b, train a **soft** voting ensemble classifier. Note that you should not use any threshold for the base models, but you should **tune the threshold** for the soft voting ensemble. Use `cv=5` while tuning the threshold. **(10 points)**

Print the best CV accuracy with a CV recall above 60%, along with the threshold that returns these results. **(2 points)**

In [24]:
probs_rf = cross_val_predict(rf_class, X_train_cleaned, y_train, cv=5, method='predict_proba')[:, 1]
probs_lgbm = cross_val_predict(lgbm_class, X_train_cleaned, y_train, cv=5, method='predict_proba')[:, 1]
probs_cat = cross_val_predict(cat_class, X_train_cleaned, y_train, cv=5, method='predict_proba')[:, 1]

avg_cv_probs = (probs_rf + probs_lgbm + probs_cat) / 3
thrs = np.arange(0.01, 1.01, 0.01)
cv_results = pd.DataFrame(columns=['threshold', 'accuracy', 'recall'])
counter = 0

for thr in thrs:
    y_pred_cv = (avg_cv_probs > thr).astype(int)
    acc = accuracy_score(y_train, y_pred_cv)
    rec = recall_score(y_train, y_pred_cv)
    cv_results.loc[counter] = [thr, acc, rec]
    counter += 1

high_recalls = cv_results[cv_results['recall'] >= 0.6]
best_threshold = high_recalls.loc[high_recalls['accuracy'].idxmax(), 'threshold']
print("best threshold:", best_threshold)
print("best cv accuracy:", high_recalls['accuracy'].max())


best threshold: 0.23
best cv accuracy: 0.7489428571428571


### d)

Using the trained soft voting ensemble classifier and the tuned threshold from Part c, print the test accuracy and the test recall. **(2 points)**

In [25]:
avg_proba = (y_pred_proba_rf + y_pred_proba_lgbm + y_pred_proba_cat) / 3
y_pred_final = (avg_proba > best_threshold).astype(int)
print("test accuracy:", accuracy_score(y_test, y_pred_final))
print("test recall:", recall_score(y_test, y_pred_final))

test accuracy: 0.7499510858931716
test recall: 0.6290050590219224


### e)

Using the same base models as in Part b, train a stacking ensemble classifier. Tune both the classifier and the threshold to get above a **(5-fold) CV acccuracy of 70% and a CV recall of 60%**. Print the hyperparameter and the threshold values that return the desired results, along with their CV accuracy and recall scores. 

Use `random_state=1` in all objects that take a `random_state`. The meta model, its hyperparameters to tune and the cross-validation setup/settings (except for the number of folds given above) are entirely up to you. 

**(13 points)**

In [26]:
# use cross validated probabilities created earlier
stack_class_df = pd.DataFrame({
    "rf_proba": probs_rf,
    "lgbm_proba": probs_lgbm,
    "cat_proba": probs_cat
})

meta_model_class = LogisticRegression(random_state = 1, solver = "liblinear")
stacking_class_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
gscv_class = GridSearchCV(
    meta_model_class,
    stacking_class_grid,
    cv = cv,
    refit = True,
    scoring = 'accuracy',
    n_jobs = -1
)
gscv_class.fit(stack_class_df, y_train)
print("best C:", gscv_class.best_params_['C'])
print("best CV accuracy:", gscv_class.best_score_)

best C: 1
best CV accuracy: 0.8849714285714286


In [27]:
tuned_stack_class = gscv_class.best_estimator_
y_pred_stack_class = cross_val_predict(tuned_stack_class, stack_class_df, y_train, cv=cv, method = "predict_proba")[:, 1]

thrs = np.arange(0.01, 1.01, 0.01)
cv_results_stack_class = pd.DataFrame(columns=['threshold', 'accuracy', 'recall'])
counter = 0

for thr in thrs:
    y_pred_cv_stack_class = (y_pred_stack_class > thr).astype(int)
    acc = accuracy_score(y_train, y_pred_cv_stack_class)
    rec = recall_score(y_train, y_pred_cv_stack_class)
    cv_results_stack_class.loc[counter] = [thr, acc, rec]
    counter += 1

high_recalls_stack_class = cv_results_stack_class[cv_results_stack_class['recall'] >= 0.6]
best_threshold_stack_class = high_recalls_stack_class.loc[high_recalls_stack_class['accuracy'].idxmax(), 'threshold']
best_accuracy = high_recalls_stack_class['accuracy'].max()
recall_best_threshold = high_recalls_stack_class.loc[high_recalls_stack_class['accuracy'].idxmax(), 'recall']
print("best threshold for stacking classifier:", best_threshold_stack_class)
print("best cv accuracy for stacking classifier:", best_accuracy)
print("recall for best threshold:", recall_best_threshold)


best threshold for stacking classifier: 0.1
best cv accuracy for stacking classifier: 0.7365428571428572
recall for best threshold: 0.6257309941520468


### f)

Using the tuned classifier and the tuned threshold in Part e, print the test accuracy and the test recall. **(2 points)**

In [28]:
stack_class_test_df = pd.DataFrame({
    "rf_proba": y_pred_proba_rf,
    "lgbm_proba": y_pred_proba_lgbm,
    "cat_proba": y_pred_proba_cat
})
y_pred_proba_stack_class_test = tuned_stack_class.predict_proba(stack_class_test_df)[:, 1]
y_pred_final_stack_class = (y_pred_proba_stack_class_test > best_threshold_stack_class).astype(int)
print("accuracy score:", accuracy_score(y_test, y_pred_final_stack_class))
print("recall score:", recall_score(y_test, y_pred_final_stack_class))

accuracy score: 0.7395812952455488
recall score: 0.657672849915683


### g)

Return the weights/importances of the base models in the Stacking Classifier from Part e. **(2 points)** Which base model seems to be the most important for the ensemble? **(1 point)**

In [29]:
model_e_imps = pd.DataFrame({
    'model': ['rf', 'lgbm', 'cat'],
    'importance': tuned_stack_class.coef_[0]
})
print(model_e_imps)

  model  importance
0    rf    0.256750
1  lgbm    2.498588
2   cat    2.605556


Catboost seems to be the most important for the ensemble, closely followed by LightGBM.